# Clasificador de perros y gatos

Preparamos el conjunto Dogs vs Cats y creamos generadores de Keras sin cargar todas las imágenes en memoria.

In [2]:
from pathlib import Path
import shutil
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
DATA_URL = "https://storage.googleapis.com/datascience-materials/dogs-vs-cats.zip"
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
ZIP_PATH = DATA_DIR / "dogs-vs-cats.zip"
RAW_DIR = DATA_DIR / "dogs-vs-cats"
SPLIT_DIR = DATA_DIR / "processed" / "dogs-vs-cats"
RAW_TRAIN_DIR = RAW_DIR / "train"

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Directorio: {PROJECT_ROOT.resolve()}")
print(f"Tamaño objetivo: {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} píxeles")

Directorio: /workspaces/machine-learning-python-template-Miguel-Contreras
Tamaño objetivo: 224x224 píxeles


## 1. Descargar y organizar el dataset

Se crea la estructura `train/test` con una subcarpeta por clase, necesaria para `flow_from_directory()`.

In [3]:
if not RAW_TRAIN_DIR.exists():
    if not ZIP_PATH.exists():
        print("Descargando el dataset...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    print("Descomprimiendo el dataset...")
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(RAW_DIR)

image_files = sorted(RAW_TRAIN_DIR.glob("*.jpg"))
if not image_files:
    raise FileNotFoundError(f"No hay imágenes JPG en {RAW_TRAIN_DIR}")

if not (SPLIT_DIR / "train" / "cat").exists():
    for split in ("train", "test"):
        for label in ("cat", "dog"):
            (SPLIT_DIR / split / label).mkdir(parents=True, exist_ok=True)
    rng = tf.random.Generator.from_seed(SEED)
    order = rng.uniform(shape=(len(image_files),), maxval=1.0).numpy().argsort()
    split_index = int(len(image_files) * 0.8)
    for position, index in enumerate(order):
        source = image_files[index]
        label = "cat" if source.name.startswith("cat.") else "dog"
        split = "train" if position < split_index else "test"
        shutil.copy2(source, SPLIT_DIR / split / label / source.name)

print(f"Imágenes originales: {len(image_files)}")
for split in ("train", "test"):
    counts = {label: len(list((SPLIT_DIR / split / label).glob("*.jpg"))) for label in ("cat", "dog")}
    print(f"{split}: {counts}")

Descargando el dataset...
Descomprimiendo el dataset...


FileNotFoundError: No hay imágenes JPG en /workspaces/machine-learning-python-template-Miguel-Contreras/data/dogs-vs-cats/train

## 2. Visualizar nueve imágenes de cada clase

Las imágenes originales tienen tamaños y proporciones diferentes; el generador las normalizará a 200×200.

In [ ]:
def show_examples(label, count=9):
    files = sorted((SPLIT_DIR / "train" / label).glob("*.jpg"))[:count]
    fig, axes = plt.subplots(3, 3, figsize=(9, 9))
    for axis, image_path in zip(axes.flat, files):
        axis.imshow(plt.imread(image_path))
        axis.set_title(image_path.name)
        axis.axis("off")
    fig.suptitle(f"Ejemplos de {label}s", fontsize=16)
    plt.tight_layout()
    plt.show()

show_examples("dog")
show_examples("cat")

## 3. Crear los generadores de Keras

`flow_from_directory()` carga por lotes, redimensiona a 200×200 y asigna las etiquetas según la carpeta.

In [ ]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)
test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)

trdata = train_datagen.flow_from_directory(
    SPLIT_DIR / "train",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=["cat", "dog"],
    shuffle=True,
    seed=SEED,
)
tsdata = test_datagen.flow_from_directory(
    SPLIT_DIR / "test",
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    classes=["cat", "dog"],
    shuffle=False,
)

print("Mapeo de clases:", trdata.class_indices)
print("Lotes de entrenamiento:", len(trdata))
print("Lotes de prueba:", len(tsdata))

In [ ]:
images, labels = next(trdata)
print("Forma del lote de imágenes:", images.shape)
print("Forma del lote de etiquetas:", labels.shape)
print("Rango de píxeles:", images.min(), "a", images.max())

## 4. Construir y entrenar la RNA

EfficientNet-B0 extrae las características visuales. `GlobalAveragePooling2D` reduce los mapas de características y las capas densas producen la clasificación binaria.

In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights=None,
    input_shape=(*IMAGE_SIZE, 3),
)
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
history = model.fit(
    trdata,
    validation_data=tsdata,
    epochs=10,
)
test_loss, test_accuracy = model.evaluate(tsdata, verbose=0)
print(f"Pérdida en prueba: {test_loss:.4f}")
print(f"Precisión en prueba: {test_accuracy:.4%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="entrenamiento")
axes[0].plot(history.history["val_accuracy"], label="validación")
axes[0].set_title("Precisión")
axes[0].legend()
axes[1].plot(history.history["loss"], label="entrenamiento")
axes[1].plot(history.history["val_loss"], label="validación")
axes[1].set_title("Pérdida")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Optimizar el entrenamiento y hacer predicciones

`ModelCheckpoint` conserva el modelo con mejor validación y `EarlyStopping` detiene el entrenamiento si deja de mejorar. En versiones actuales de Keras se usa `fit`; es el reemplazo de `fit_generator`.

In [ ]:
ModelCheckpoint = tf.keras.callbacks.ModelCheckpoint
EarlyStopping = tf.keras.callbacks.EarlyStopping

checkpoint_path = PROJECT_ROOT / "models" / "efficientnet_b0_best.keras"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
checkpoint = ModelCheckpoint(
    filepath=checkpoint_path,
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1,
)
early_stopping = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=3,
    restore_best_weights=True,
    verbose=1,
)

In [ ]:
optimized_history = model.fit(
    trdata,
    validation_data=tsdata,
    epochs=30,
    callbacks=[checkpoint, early_stopping],
)
best_model = tf.keras.models.load_model(checkpoint_path)
test_loss, test_accuracy = best_model.evaluate(tsdata, verbose=0)
print(f"Mejor modelo - pérdida en prueba: {test_loss:.4f}")
print(f"Mejor modelo - precisión en prueba: {test_accuracy:.4%}")

In [ ]:
import numpy as np
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

tsdata.reset()
probabilities = best_model.predict(tsdata, verbose=1)
predicted_classes = np.argmax(probabilities, axis=1)
matrix = confusion_matrix(tsdata.classes, predicted_classes)
ConfusionMatrixDisplay(matrix, display_labels=list(tsdata.class_indices)).plot(cmap="Blues")
plt.title("Matriz de confusión del mejor modelo")
plt.show()

## 6. Guardar el modelo optimizado

In [ ]:
final_model_path = PROJECT_ROOT / "models" / "dogs_vs_cats_efficientnet_b0.keras"
best_model.save(final_model_path)
print(f"Modelo guardado en: {final_model_path.resolve()}")